# Use cases

It's nice that someone has built a new stream processing library. But why? For what?

This chapter helps you find inspiration.

It starts with an example for [fan-out](#fan_out). We read a source stream, do something and "fan-out" to multiple sink streams.


## Overview

[Preparation](#prep)

* [Fan-out](#fan_out)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
import sys
sys.path.insert(1, ".")
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "order"
small_sink_str = "small"
medium_sink_str = "medium"
large_sink_str = "large"


---
<a id="fan_out"></a>
## Fan-out

We read a source stream of orders and fan-out depending on the price of the order:
* each order with `price <= 33` goes to the sink for small price orders
* each order with `price > 33 and price <= 66` goes to the sink for medium price orders
* each order with `price > 66` goes to the sink for large price orders

Here is the code.

In [ ]:
source_tn = Tn.source(order_source_str)

small_sink_tn = (
    source_tn
    .filter(lambda r: r["value"]["price"] <= 33)
    .peek("small")
    .sink(small_sink_str)
)

medium_sink_tn = (
    source_tn
    .filter(lambda r: r["value"]["price"] > 33 and r["value"]["price"] <= 66)
    .peek("medium")
    .sink(medium_sink_str)
)

large_sink_tn = (
    source_tn
    .filter(lambda r: r["value"]["price"] > 66)
    .peek("large")
    .sink(large_sink_str)
)
#
tn = Tn.build(small_sink_tn, medium_sink_tn, large_sink_tn)
#
for _ in range(10):
    m_list = order_generator.generate(10)
    #
    tn.process({order_source_str:  m_list})

